In [1]:
from collections import Counter
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
import os 
from dotenv import load_dotenv

import csv
from typing import List, Dict
from trulens.apps.custom import TruCustomApp
from trulens.core import TruSession
from trulens.core import Feedback
from trulens.providers.openai import OpenAI as tru_openai
from trulens.apps.custom import instrument


In [2]:
load_dotenv()

True

In [ ]:
from trulens.core import TruSession


session = TruSession()

# Uncomment the following to reset database 
# session.reset_database()

In [ ]:
client = OpenAI()

In [ ]:
pc = Pinecone(api_key= os.getenv("PINECONE_API_KEY_500"))
index_name = "sentence-window-retrieval"

In [ ]:
indices = []
for index in pc.list_indexes():
    indices.append(index["name"])

if index_name in indices :
    print(f"index {index_name} already exists!")
    index = pc.Index(index_name)
else:
    pc.create_index(
  name=index_name,
  dimension=3072,
  metric="dotproduct",
  spec=ServerlessSpec(
    cloud="aws",
    region="us-east-1"
  ),
  deletion_protection="disabled"
)
    index = pc.Index(index_name)
    print(f"index {index_name} created")



In [4]:
from llama_index.core.node_parser import SentenceWindowNodeParser
from llama_index.core import Document

node_parser = SentenceWindowNodeParser.from_defaults(
    # how many sentences on either side to capture
    window_size=3,
    # the metadata key that holds the window of surrounding sentences
    window_metadata_key="window",
    # the metadata key that holds the original sentence
    original_text_metadata_key="original_sentence",
)

###PREPARE WEBPAGE DATA 
###Get the full text & Chunks
import json 

with open(r"../../data/url_content_mapping.json", "r",encoding="utf-8") as file:
    data = json.load(file)
texts = []
for content in data:
    texts.extend(node_parser.get_nodes_from_documents([Document(text=content["content"])]))


In [15]:
print(f"""{texts[0].__dir__()}
{texts[0].metadata["window"]}
{len(texts[0].metadata["window"])}
{texts[0].metadata["original_sentence"]}
{len(texts[0].metadata["original_sentence"])}
{texts[0].text}
{len(texts[0].text)}
{len(texts)}""")

['id_', 'embedding', 'metadata', 'excluded_embed_metadata_keys', 'excluded_llm_metadata_keys', 'relationships', 'metadata_template', 'metadata_separator', 'text', 'mimetype', 'start_char_idx', 'end_char_idx', 'metadata_seperator', 'text_template', '__module__', '__annotations__', '__doc__', '__init__', 'class_name', 'hash', 'get_type', 'get_content', 'get_metadata_str', 'set_content', 'get_node_info', 'get_text', 'node_info', 'model_config', '__class_vars__', '__private_attributes__', '__abstractmethods__', '_abc_impl', '__pydantic_custom_init__', '__pydantic_post_init__', '__pydantic_decorators__', '__pydantic_generic_metadata__', '__pydantic_complete__', '__pydantic_parent_namespace__', '__pydantic_fields__', '__pydantic_core_schema__', '__pydantic_validator__', '__pydantic_serializer__', '__signature__', '__pydantic_computed_fields__', 'node_id', 'source_node', 'prev_node', 'next_node', 'parent_node', 'child_nodes', 'ref_doc_id', 'extra_info', '__str__', 'get_embedding', 'as_related